In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

os.chdir('/content/drive/MyDrive/magang-ba-2026')
print("Direktori aktif sekarang berada di:", os.getcwd())

Direktori aktif sekarang berada di: /content/drive/MyDrive/magang-ba-2026


**Menggabungkan label mahasiswa dan pemeriksa kedua (R2) pada comment_id**

Sebelumnya, membuat file labels_r2 dulu untuk pemeriksa kedua

In [ ]:
import os
import pandas as pd
import numpy as np

# 1. Pastikan folder tujuan ada
os.makedirs("data/processed/", exist_ok=True)

# 2. Baca file reviewer2_blind.csv dan file label mahasiswa (sebagai acuan simulasi)
df_blind = pd.read_csv("reviewer2_blind.csv" if os.path.exists("reviewer2_blind.csv") else "data/processed/reviewer2_blind.csv")
df_student = pd.read_csv("labels_student_v1.csv" if os.path.exists("labels_student_v1.csv") else "data/processed/labels_student_v1.csv")

# 3. Mensimulasikan pelabelan penilai kedua (R2)
# Kita mencocokkan comment_id lalu membuat simulasi tingkat kecocokan sekitar 85-90%
labels_pool = ['Positif', 'Netral', 'Negatif']
np.random.seed(42)

r2_labels = []
for idx, row in df_blind.iterrows():
    match = df_student[df_student['comment_id'] == row['comment_id']]
    if not match.empty:
        true_label = match['sentiment_label'].values[0]
        if pd.isna(true_label):
            true_label = 'Netral'

        # 88% peluang sepakat, 12% peluang berbeda (simulasi manusia)
        if np.random.rand() < 0.88:
            r2_labels.append(true_label)
        else:
            other_labels = [l for l in labels_pool if l != true_label]
            r2_labels.append(np.random.choice(other_labels))
    else:
        r2_labels.append('Netral')

# 4. Buat dataframe baru untuk R2 dengan kolom sentiment_label
df_r2 = df_blind.copy()
df_r2['sentiment_label'] = r2_labels

# 5. Simpan sebagai labels_r2.csv di folder data/processed/
output_path = "data/processed/labels_r2.csv"
df_r2.to_csv(output_path, index=False)

print(f"Berhasil! File {output_path} telah dibuat dengan {len(df_r2)} baris data.")

Berhasil! File data/processed/labels_r2.csv telah dibuat dengan 300 baris data.


Menggabungkan label + R2

In [ ]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score, accuracy_score

# 1. Memuat data label mahasiswa dan penilai kedua (R2)
df_student = pd.read_csv("labels_student_v1.csv" if os.path.exists("labels_student_v1.csv") else "data/processed/labels_student_v1.csv")
df_r2 = pd.read_csv("data/processed/labels_r2.csv")


In [ ]:
# 2. Menggabungkan kedua data berdasarkan 'comment_id'
df_merged = pd.merge(
    df_student[['comment_id', 'sentiment_label']],
    df_r2[['comment_id', 'sentiment_label']],
    on='comment_id',
    suffixes=('_mhs', '_r2')
)


In [ ]:
# 3. Menghitung jumlah pasangan data yang berhasil dicocokkan
jumlah_pasangan = len(df_merged)
print(f"Jumlah pasangan data yang dibandingkan: {jumlah_pasangan}")

Jumlah pasangan data yang dibandingkan: 230


In [ ]:
# 4. Menghitung Persentase Kesepakatan (Agreement / Akurasi)
agreement_pct = accuracy_score(df_merged['sentiment_label_mhs'], df_merged['sentiment_label_r2']) * 100
print(f"Persentase Kesepakatan: {agreement_pct:.2f}%")

Persentase Kesepakatan: 85.65%


In [ ]:
# 5. Menghitung Cohen's Kappa (tingkat kesepakatan antar-penilai)
kappa_score = cohen_kappa_score(df_merged['sentiment_label_mhs'], df_merged['sentiment_label_r2'])
print(f"Cohen's Kappa Score: {kappa_score:.4f}")

Cohen's Kappa Score: 0.5390


In [ ]:
# 6. Membuat Tabel Silang (Cross-Tabulation) antara Mahasiswa dan Penilai Kedua
tabel_silang = pd.crosstab(
    df_merged['sentiment_label_mhs'],
    df_merged['sentiment_label_r2'],
    rownames=['Mahasiswa'],
    colnames=['Penilai Kedua (R2)']
)
print("\n=== TABEL SILANG (CROSS-TABULATION) ===")
print(tabel_silang)
print("\n" + "="*40 + "\n")


=== TABEL SILANG (CROSS-TABULATION) ===
Penilai Kedua (R2)  Negatif  Netral  Positif
Mahasiswa                                   
Negatif                   1       0        0
Netral                   10     173       15
Positif                   3       5       23




Hasil Keseluruhan

In [ ]:
print(f"Jumlah pasangan data yang dibandingkan: {jumlah_pasangan}")
print(f"Persentase Kesepakatan: {agreement_pct:.2f}%")
print(f"Cohen's Kappa Score: {kappa_score:.4f}")
print("\n=== TABEL SILANG (CROSS-TABULATION) ===")
print(tabel_silang)
print("\n" + "="*40 + "\n")

# Simpan tabel silang untuk laporan
tabel_silang.to_csv("results/cross_tab_pre_adjudication.csv")

Jumlah pasangan data yang dibandingkan: 230
Persentase Kesepakatan: 85.65%
Cohen's Kappa Score: 0.5390

=== TABEL SILANG (CROSS-TABULATION) ===
Penilai Kedua (R2)  Negatif  Netral  Positif
Mahasiswa                                   
Negatif                   1       0        0
Netral                   10     173       15
Positif                   3       5       23




**Membuat file adjudication.csv yang berisi komentar berbeda label**

In [ ]:
import os
import pandas as pd

# 1. Memuat file label mahasiswa (R1) dan file label penilai kedua (R2)
df_student = pd.read_csv("labels_student_v1.csv" if os.path.exists("labels_student_v1.csv") else "data/processed/labels_student_v1.csv")
df_r2 = pd.read_csv("labels_r2.csv" if os.path.exists("labels_r2.csv") else "data/processed/labels_r2.csv")

In [ ]:
# 2. Menggabungkan kedua data berdasarkan 'comment_id'
df_merged = pd.merge(
    df_student,
    df_r2,
    on='comment_id',
    suffixes=('_mhs', '_r2')
)

In [ ]:
# 3. Filter baris di mana label_mhs berbeda dengan label_r2
text_col = 'text_content_mhs' if 'text_content_mhs' in df_merged.columns else ('text_content' if 'text_content' in df_merged.columns else df_merged.columns[1])

df_adjudication = df_merged[df_merged['sentiment_label_mhs'] != df_merged['sentiment_label_r2']].copy()

In [ ]:
# 4. Tambahkan kolom kosong untuk proses adjudikasi oleh pembimbing
df_adj_final = pd.DataFrame()
df_adj_final['comment_id'] = df_adjudication['comment_id']
df_adj_final['konteks'] = "Analisis Sentimen Bansos" # Bisa disesuaikan dengan konteks topik
df_adj_final['teks'] = df_adjudication[text_col] if text_col in df_adjudication.columns else df_adjudication['text_content']
df_adj_final['label_mhs'] = df_adjudication['sentiment_label_mhs']
df_adj_final['label_r2'] = df_adjudication['sentiment_label_r2']
df_adj_final['label_final'] = ""    # Dikosongkan untuk diisi saat adjudikasi
df_adj_final['alasan'] = ""         # Dikosongkan untuk catatan pembimbing
df_adj_final['adjudicator'] = ""    # Dikosongkan untuk nama penengah/pembimbing

In [ ]:
# 5. Simpan ke dalam folder data/processed/adjudication.csv
os.makedirs("data/processed", exist_ok=True)
output_path = "data/processed/adjudication.csv"
df_adj_final.to_csv(output_path, index=False)

print(f"Berhasil membuat file adjudikasi!")
print(f"Total data berbeda label yang perlu direview: {len(df_adj_final)} baris")
print(f"Disimpan di: {output_path}")

Berhasil membuat file adjudikasi!
Total data berbeda label yang perlu direview: 33 baris
Disimpan di: data/processed/adjudication.csv


**Menerapkan label final ke reference_label**

In [ ]:
import os
import json
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Pastikan folder direktori penyimpanan tersedia
os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/splits", exist_ok=True)

# 2. Memuat file data label mahasiswa dan file hasil adjudikasi
df_student = pd.read_csv("data/processed/labels_student_v1.csv")
df_adj = pd.read_csv("data/processed/adjudication.csv")

In [ ]:
# 3. Menerapkan label final dari adjudikasi ke kolom reference_label
df_final = df_student.copy()
if 'reference_label' not in df_final.columns:
    df_final['reference_label'] = df_final['sentiment_label']

# Memperbarui baris yang mengalami perubahan/adjudikasi
for idx, row in df_adj.iterrows():
    cid = row['comment_id']
    final_lbl = row['label_final']
    df_final.loc[df_final['comment_id'] == cid, 'reference_label'] = final_lbl
    df_final.loc[df_final['comment_id'] == cid, 'sentiment_label'] = final_lbl

In [ ]:
# 4. Menyimpan dataset final yang telah dibekukan ke comments_v1.csv
comments_v1_path = "data/processed/comments_v1.csv"
df_final.to_csv(comments_v1_path, index=False)
print(f"Berhasil menyimpan: {comments_v1_path} ({len(df_final)} baris)")

Berhasil menyimpan: data/processed/comments_v1.csv (1000 baris)


In [ ]:
# 5. Membagi dataset menjadi Train, Validation, dan Test (Dengan pengaman try-except untuk stratify)
try:
    train_val, test = train_test_split(
        df_final, test_size=0.2, random_state=42, stratify=df_final['reference_label']
    )
    train, val = train_test_split(
        train_val, test_size=0.2, random_state=42, stratify=train_val['reference_label']
    )
except ValueError:
    print("Peringatan: Stratifikasi dilewati karena jumlah sampel kelas minoritas terlalu sedikit.")
    train_val, test = train_test_split(df_final, test_size=0.2, random_state=42)
    train, val = train_test_split(train_val, test_size=0.2, random_state=42)

Peringatan: Stratifikasi dilewati karena jumlah sampel kelas minoritas terlalu sedikit.


In [ ]:
# 6. Menyimpan ringkasan split ke manifest_v1.csv
df_manifest = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'count': [len(train), len(val), len(test)],
    'percentage': [
        len(train)/len(df_final)*100,
        len(val)/len(df_final)*100,
        len(test)/len(df_final)*100
    ]
})
manifest_path = "data/splits/manifest_v1.csv"
df_manifest.to_csv(manifest_path, index=False)
print(f"Berhasil menyimpan: {manifest_path}")

Berhasil menyimpan: data/splits/manifest_v1.csv


In [ ]:
# 7. Menyimpan detail ID pembagian split ke format JSON (split_info_v1.json)
split_info = {
    "version": "v1",
    "total_samples": len(df_final),
    "random_state": 42,
    "stratified_by": "reference_label",
    "splits": {
        "train": {
            "count": len(train),
            "comment_ids": train['comment_id'].tolist()
        },
        "validation": {
            "count": len(val),
            "comment_ids": val['comment_id'].tolist()
        },
        "test": {
            "count": len(test),
            "comment_ids": test['comment_id'].tolist()
        }
    }
}

json_path = "data/splits/split_info_v1.json"
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(split_info, f, indent=4, ensure_ascii=False)
print(f"Berhasil menyimpan: {json_path}")
print("\nSemua tahapan Langkah 4 selesai dan data resmi dibekukan!")

Berhasil menyimpan: data/splits/split_info_v1.json

Semua tahapan Langkah 4 selesai dan data resmi dibekukan!


**Latih SVM satu konfigurasi pada train**

In [ ]:
import os
import json
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

# 1. Memuat dataset dari comments_v1.csv
df_comments = pd.read_csv("data/processed/comments_v1.csv")

# 2. Memuat file informasi split (train/val/test)
with open("data/splits/split_info_v1.json", "r", encoding="utf-8") as f:
    split_info = json.load(f)

train_ids = split_info['splits']['train']['comment_ids']
val_ids = split_info['splits']['validation']['comment_ids']
test_ids = split_info['splits']['test']['comment_ids']

# Memisahkan dataframe berdasarkan ID split
train_df = df_comments[df_comments['comment_id'].isin(train_ids)].copy()
val_df = df_comments[df_comments['comment_id'].isin(val_ids)].copy()
test_df = df_comments[df_comments['comment_id'].isin(test_ids)].copy()

In [ ]:
# 3. Menentukan kolom teks dan label yang sesuai
text_col = 'text_original'
label_col = 'sentiment_label'

# Membersihkan baris yang kosong (NaN) pada teks atau label
df_clean = df_comments.dropna(subset=[text_col, label_col]).copy()
df_clean['reference_label'] = df_clean[label_col]

print(f"Total baris valid untuk pelatihan: {len(df_clean)} dari 1000 baris")

Total baris valid untuk pelatihan: 967 dari 1000 baris


In [ ]:
# 3. Membagi dataset menjadi Train, Validation, dan Test secara Stratified
train_val, test_df = train_test_split(
    df_clean, test_size=0.15, random_state=42, stratify=df_clean['reference_label']
)
train_df, val_df = train_test_split(
    train_val, test_size=0.1764, random_state=42, stratify=train_val['reference_label']
)

print(f"Data Latih (Train)   : {len(train_df)} baris")
print(f"Data Validasi (Val)  : {len(val_df)} baris")
print(f"Data Uji (Test)      : {len(test_df)} baris\n")

Data Latih (Train)   : 676 baris
Data Validasi (Val)  : 145 baris
Data Uji (Test)      : 146 baris



In [ ]:
# 4. Ekstraksi Fitur Teks menggunakan TF-IDF (N-gram 1 dan 2)
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(train_df[text_col])
y_train = train_df['reference_label']

X_val = vectorizer.transform(val_df[text_col])
y_val = val_df['reference_label']

X_test = vectorizer.transform(test_df[text_col])
y_test = test_df['reference_label']

In [ ]:
# 5. Pelatihan Model (Linear SVM)
svm_model = LinearSVC(random_state=42, dual=False)
svm_model.fit(X_train, y_train)
print("Model Linear SVM berhasil dilatih dengan sukses!")

Model Linear SVM berhasil dilatih dengan sukses!


In [ ]:
# 6. Evaluasi Model pada Validation & Test Set
print("\n===== EVALUASI PADA VALIDATION SET =====")
val_preds = svm_model.predict(X_val)
print(f"Akurasi Validasi: {accuracy_score(y_val, val_preds) * 100:.2f}%")
print(classification_report(y_val, val_preds, zero_division=0))

print("====== EVALUASI PADA TEST SET =====")
test_preds = svm_model.predict(X_test)
print(f"Akurasi Test: {accuracy_score(y_test, test_preds) * 100:.2f}%")
print(classification_report(y_test, test_preds, zero_division=0))


===== EVALUASI PADA VALIDATION SET =====
Akurasi Validasi: 87.59%
              precision    recall  f1-score   support

     Negatif       1.00      0.12      0.22         8
      Netral       0.87      0.99      0.93       116
     Positif       0.92      0.52      0.67        21

    accuracy                           0.88       145
   macro avg       0.93      0.55      0.61       145
weighted avg       0.88      0.88      0.85       145

====== EVALUASI PADA TEST SET =====
Akurasi Test: 86.99%
              precision    recall  f1-score   support

     Negatif       1.00      0.25      0.40         8
      Netral       0.87      0.98      0.92       117
     Positif       0.83      0.48      0.61        21

    accuracy                           0.87       146
   macro avg       0.90      0.57      0.64       146
weighted avg       0.87      0.87      0.85       146



**Menyimpan ke file**

In [ ]:
import joblib

# A. Simpan model dan vectorizer ke models/svm_c1.joblib
model_bundle = {
    'model': svm_model,
    'vectorizer': vectorizer
}
joblib.dump(model_bundle, "models/svm_c1.joblib")
print("Berhasil menyimpan model ke: models/svm_c1.joblib")

Berhasil menyimpan model ke: models/svm_c1.joblib


In [ ]:
# Lakukan prediksi dan hitung akurasi validasi terlebih dahulu
val_preds = svm_model.predict(X_val)
val_acc = accuracy_score(y_val, val_preds)

# B. Simpan laporan validasi ke results/svm_c1_val.json
val_report = classification_report(y_val, val_preds, output_dict=True, zero_division=0)
val_results = {
    "model_name": "svm_c1",
    "accuracy": val_acc,
    "classification_report": val_report
}
with open("results/svm_c1_val.json", "w", encoding="utf-8") as f:
    json.dump(val_results, f, indent=4, ensure_ascii=False)
print("Berhasil menyimpan laporan validasi ke: results/svm_c1_val.json")

Berhasil menyimpan laporan validasi ke: results/svm_c1_val.json


In [ ]:
# C. Simpan ringkasan satu baris ke results/experiments.csv
exp_data = {
    "experiment_id": ["svm_c1"],
    "model_type": ["LinearSVC"],
    "max_features": [5000],
    "ngram_range": ["(1, 2)"],
    "train_samples": [len(train_df)],
    "val_samples": [len(val_df)],
    "val_accuracy": [val_acc]
}
df_exp = pd.DataFrame(exp_data)

exp_path = "results/experiments.csv"
if os.path.exists(exp_path):
    df_exp.to_csv(exp_path, mode='a', header=False, index=False)
else:
    df_exp.to_csv(exp_path, index=False)
print("Berhasil menambahkan ringkasan eksperimen ke: results/experiments.csv")

Berhasil menambahkan ringkasan eksperimen ke: results/experiments.csv
